### import env var and libraries

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
import gradio as gr
import json


load_dotenv()

client = OpenAI()

/home/pouya/Documents/AI-Engineer-SDS/ai_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### set up Pushover

In [2]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"


In [3]:
#Test pushover
import requests

def send_notification(message: str):
    payload = { "user": pushover_user, "token": pushover_token, "message": message }
    response = requests.post(pushover_url, data=payload)
    return response

In [4]:
send_notification("Hello from VScode!")

<Response [200]>

### describe pushover as an LLM tool

In [5]:
send_notification_function = {
    "name": "send_notification",
    "description": "Sends a pushover notification to the user's phone via the pushover API. Use this to alert the user about important information.",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The message to send in the notification."
             }
            },
        "required": ["message"]
        }
}

### add pushover to the list of tools for LLM

In [6]:
tools = [{"type": "function", "function": send_notification_function}]

### Create new function, describe it, and add it to the list of tools

In [ ]:
import random

#simulates rolling a 6-sided dice
def dice_roll():
    result = random.randint(1,6)
    return result

# describe function for LLM
roll_dice_function = {
    "name": "dice_roll",
    "description": "Simulate a 6-sided dice roll for the user",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The message to send in the notification."
             }
            },
        "required": ["message"]
        }
} 

#Add function to list of tools LLM can use
tools = [{"type": "function", "function": send_notification_function}, {"type": "function", "function": roll_dice_function}]

### Calling the tool from an LLM

In [7]:
from litellm import completion
client = OpenAI()
response = completion(
    model="gpt-4.1-mini",
    messages = [{"role": "user", "content": "Send me a push notification to my phone of a fun fact about software engineering someone who is \
    trying to land an intern role would like to hear (context: I am a first year Data Scienc major in SJSU)"}],
    tools=tools,
    tool_choice="auto"
)
message = response.choices[0].message

print(message)

Message(content=None, role='assistant', tool_calls=[ChatCompletionMessageToolCall(function=Function(arguments='{"message":"Fun Fact: Did you know that one of the first versions of the term \'bug\' in software engineering came from a real moth found causing issues in an early computer? Debugging is literally making the code bug-free!"}', name='send_notification'), id='call_bk88iBPXvjomvYt8wRU89o2H', type='function')], function_call=None, provider_specific_fields={'refusal': None}, annotations=[])


In [8]:
if message.tool_calls: 
    tool_call = message.tool_calls[0]
    args = json.loads(tool_call.function.arguments)
    
    send_notification(args['message'])
    print("Notification sent!")
    
else:
    print(message.content)


Notification sent!


### LLM tool calling function (unified)

In [9]:
from litellm import completion

client = OpenAI()

send_notification_function = {
    "name": "send_notification",
    "description": "Sends a pushover notification to the user's phone via the pushover API. Use this to alert the user about important information.",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The message to send in the notification."
             }
            },
        "required": ["message"]
        }
}

tools = [{"type": "function", "function": send_notification_function}]

prompt = [{"role": "user", "content": "Send me a push notification to my phone of a fun fact about software/ai engineering \
            for someone who is trying to land an intern role in the next 3-6 months. keep it 1-2.5 sentances \
            (context: I am a first year Data Scienc major in SJSU)"}]

response = completion(
    model="gpt-4.1-mini",
    messages = prompt,
    tools=tools,
    temperature= 1.1,
    tool_choice="auto"
)
message = response.choices[0].message

if message.tool_calls: 
    tool_call = message.tool_calls[0]
    args = json.loads(tool_call.function.arguments)
    
    send_notification(args['message'])
    print("Notification sent!")
    
else:
    print(message.content)


Notification sent!


### LLM tool call handling

In [10]:
def handle_tool_call(tool_calls):
    tool_call = tool_calls[0] #assuming just one tool call
    args = json.loads(tool_call.function.arguments)

    #actually send the noti
    send_notification(args['message'])

    tool_call_result = {
        "role": "tool" ,
        "content": f"Notification sent: {args['message']}" , 
        "tool_call_id": tool_call.id
    }

    #return what to add to our "context" (about tool call results), a dictionary
    return tool_call_result

### proper tool call handling 

In [11]:
from litellm import completion
client = OpenAI()
messages = [{"role": "user", "content": "Send me a push notification to my phone of a fun fact about software engineering someone who is \
    trying to land an intern role would like to hear (context: I am a first year Data Scienc major in SJSU)"}]

response = completion(
    model="gpt-4.1-mini",
    messages = messages,
    tools=tools,
    tool_choice="auto"
)
message = response.choices[0].message


#Check if model wants to call a tool
if message.tool_calls:
    #.. handle tool call
    toolDict = handle_tool_call(message.tool_calls)

    #.. add message to context , i.e messages
    messages.append(message)

    #.. add info about tool call response to message (context)so t
    messages.append(toolDict)

    #.. invoke LLM again to get its updated response    
    response = completion(
        model="gpt-4.1-mini",
        messages = messages,
        tools=tools,
        tool_choice="auto"
    )
    message = response.choices[0].message
    
    #.. print(message.content) --- from LLM response
    print(message.content)

else:
    print(message.content)

I've sent a push notification to your phone with a fun fact about software engineering that could be interesting and motivating for someone like you aiming for a Data Science intern role at SJSU. If you'd like more facts or tips, just let me know!


### Multiple LLM tool calls

In [ ]:
from litellm import completion

def handle_tool_call(tool_calls):
    tool_results = []

    for tool_call in tool_calls:
        args = json.loads(tool_call.function.arguments)

        #actually send the noti
        send_notification(args['message'])

        tool_call_result = {
            "role": "tool" ,
            "content": f"Notification sent: {args['message']}" , 
            "tool_call_id": tool_call.id
        }
        tool_results.append(tool_call_result)

    #return what to add to our "context" (about tool call results), a dictionary
    return tool_results

client = OpenAI()
messages = [{"role": "user", "content": "Send me 2 push notification to my phone of a fun fact about software engineering someone who is \
    trying to land an intern role would like to hear (context: I am a first year Data Scienc major in SJSU)"}]

response = completion(
    model="gpt-4.1-mini",
    messages = messages,
    tools=tools,
    tool_choice="auto"
)
message = response.choices[0].message


#Check if model wants to call a tool
if message.tool_calls:
    #.. handle tool call
    toolDict = handle_tool_call(message.tool_calls)

    #.. add message to context , i.e messages
    messages.append(message)

    #.. add info about tool call response to message (context)so t
    messages.extend(toolDict)

    #.. invoke LLM again to get its updated response    
    response = completion(
        model="gpt-4.1-mini",
        messages = messages,
        tools=tools,
        tool_choice="auto"
    )
    message = response.choices[0].message
    
    #.. print(message.content) --- from LLM response
    print(message.content)

else:
    print(message.content)

None


### Future Proof function to handle different types of tools

In [ ]:
def handle_tool_call(tool_calls):
    tool_results = []

    for tool_call in tool_calls:
        function_name = tool_call.function.name 
        args = json.loads(tool_call.function.arguments)

        #Route to the appropriate function based on the function_name
        if function_name == "send_notification":
            send_notification(args['message'])
            content = f"Notification sent: {args['message']}"
        # elif function_name == "function_name_v2":
        #      call function_name_v2
        # elif function_name == "function_name_v3":
        #      call function_name_v3
        #... 
        else:
            content = f"Unkown Function: {function_name}"


        tool_call_result = {
            "role": "tool" ,
            "content": f"Notification sent: {args['message']}" , 
            "tool_call_id": tool_call.id
        }
        tool_results.append(tool_call_result)

    #return what to add to our "context" (about tool call results), a dictionary
    return tool_results